In [104]:
import pandas as pd
import numpy as np
import re
import io
from pathlib import Path

In [105]:
class DatasetLoader:
    def __init__(self, file_path):
        self.file_path = file_path
    
    def load(self):
        with open(self.file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        
        # Find the header line (robust to spacing)
        start = next(
            i for i, l in enumerate(lines)
            if l.strip().startswith("Study Subject ID")
        )
        
        # Store metadata lines (everything before the data header)
        metadata = lines[:start]
        
        # Extract data portion (from header line onwards)
        data_text = "".join(lines[start:])
        
        # Read the TSV data
        df = pd.read_csv(io.StringIO(data_text), sep="\t", dtype=str, engine="python")
        
        # Clean column names: remove whitespace and replace spaces with dots
        df.columns = [c.strip().replace(" ", ".") for c in df.columns]
        
        return df, metadata


In [106]:
# Load the main dataset
data_path = "AI-PMP_dataset_2026-04-07.tsv"
print("Loading dataset...")
loader = DatasetLoader(data_path)
df, metadata = loader.load()

print(f"Dataset shape: {df.shape}")
print(f"Number of columns: {len(df.columns)}")
print(f"Metadata lines: {len(metadata)}")
print(f"\nFirst 5 column names:")
for col in df.columns[:10]:
    print(f"  - {col}")
if len(df.columns) > 10:
    print("  ...")
print(f"\nFirst 2 rows of data:")
df.head(2)

Loading dataset...
Dataset shape: (30, 693)
Number of columns: 693
Metadata lines: 38

First 5 column names:
  - Study.Subject.ID
  - Protocol.ID
  - Subject.Status
  - Sex
  - StartDate_E1
  - EndDate_E1
  - Event.Status_E1
  - Interviewer_E1_C1
  - Interview.Date_E1_C1
  - CRF.Version.Status_E1_C1
  ...

First 2 rows of data:


,Study.Subject.ID,Protocol.ID,Subject.Status,Sex,StartDate_E1,EndDate_E1,Event.Status_E1,Interviewer_E1_C1,Interview.Date_E1_C1,CRF.Version.Status_E1_C1,...,PDQ39_32_E1_C24,PDQ39_33_E1_C24,PDQ39_34_E1_C24,PDQ39_35_E1_C24,PDQ39_36_E1_C24,PDQ39_37_E1_C24,PDQ39_38_E1_C24,PDQ39_39_E1_C24,PDQ39_TOTAL_E1_C24,Unnamed:.692
0,fr02001,AI-PMP - AI-PMP-CHUT,available,m,2025-11-24,2025-11-24,data entry started,CHUTDatEnt2,2025-11-24,data entry complete,...,3,3,2,2,0,0,2,0,42,NaN
1,fr02002,AI-PMP - AI-PMP-CHUT,available,m,2025-12-15,NaN,data entry started,CHUTDatEnt2,2025-12-15,data entry complete,...,1,1,2,1,2,1,2,1,28,NaN


In [107]:
# Load the lookup Excel file
lookup_path = "AI-PMP data cleaning May_Variables and Ranges.xlsx"
print("\nLoading lookup file...")
lookup_df_full = pd.read_excel(lookup_path, sheet_name="ALL")
print(f"Full lookup shape: {lookup_df_full.shape}")



Loading lookup file...
Full lookup shape: (174, 31)


In [108]:
# Focus on the relevant columns
lookup_df = lookup_df_full[['ITEM_NAME*', 'type_of', 'range_of']].copy()
lookup_df.columns = ['variable_base', 'type_of', 'range_of']


In [109]:
lookup_with_rules = lookup_df[
    ~pd.isna(lookup_df['type_of']) & 
    ~pd.isna(lookup_df['range_of']) &
    (lookup_df['type_of'].astype(str).str.strip() != '') &
    (lookup_df['range_of'].astype(str).str.strip() != '')
].copy()

print(f"\nLookup rows with validation rules: {len(lookup_with_rules)}")
print(f"Lookup rows without rules (skipped): {len(lookup_df) - len(lookup_with_rules)}")

print(f"\nFirst 10 validation rules from lookup:")
lookup_with_rules.head(10)


Lookup rows with validation rules: 89
Lookup rows without rules (skipped): 85

First 10 validation rules from lookup:


,variable_base,type_of,range_of
0,MI_INTER_WORS_Q,numeric.discrete,"c(1,0)"
1,MI_INTER_WORS_Y_MS_YN,numeric.discrete,"c(1,0)"
2,MI_INTER_WORS_Y_MS,numeric.discrete,"c(1,2,3,4,5,6)"
3,MI_INTER_WORS_Y_NMS_YN,numeric.discrete,"c(1,0)"
4,MI_INTER_WORS_Y_NMS,numeric.discrete,"c(1,2,3,4,5,6,7,8)"
7,MI_INTER_PD_MED_CHANGE,numeric.discrete,"c(1,0)"
8,MI_INTER_WORS_Q,numeric.discrete,"c(1,0)"
9,MI_INTER_WORS_Y_MS_YN,numeric.discrete,"c(1,0)"
10,MI_INTER_WORS_Y_MS,numeric.discrete,"c(1,2,3,4,5,6)"
11,MI_INTER_WORS_Y_NMS_YN,numeric.discrete,"c(1,0)"


In [110]:
def is_missing(x):
    """Check for missing values (NA, empty strings, whitespace only)"""
    if x.dtype == "O":
        return x.isna() | (x.astype(str).str.strip() == "")
    return x.isna()

def parse_range(range_str):
    """
    Parse R-style ranges like 'c(1,2,3)' or 'c(1935,2025)'
    PRESERVES INTEGERS - if all values are whole numbers, keep them as ints
    """
    if pd.isna(range_str):
        return None
    range_str = str(range_str)
    # Remove c() and quotes
    range_str = re.sub(r'c\(|\)|"|\'', "", range_str)
    vals = [v.strip() for v in range_str.split(",") if v.strip() != ""]
    if not vals:
        return None
    
    # Try to convert to appropriate type (int if possible, otherwise float)
    result = []
    all_ints = True
    for v in vals:
        try:
            # Check if it's a whole number
            if '.' in v:
                f = float(v)
                if f.is_integer():
                    result.append(int(f))
                else:
                    result.append(f)
                    all_ints = False
            else:
                # No decimal point, try int first
                result.append(int(v))
        except ValueError:
            # Not a number, keep as string
            result.append(v)
            all_ints = False
    
    return result

def extract_year(x):
    """Extract year from date strings (first 4 digits)"""
    return pd.to_numeric(x.astype(str).str[:4], errors="coerce")

def is_year_variable(col_name):
    """Check if column is likely a year/date variable"""
    year_keywords = ['YEAR', 'DATE', 'DOB', 'Y_', '_YEAR', 'YEAR_']
    return any(kw in col_name.upper() for kw in year_keywords)


In [111]:
def build_rules_with_prefix(df, lookup_with_rules):
    """
    Build validation rules where lookup variable_base serves as a prefix.
    Matches any column that STARTS WITH the base name (case-insensitive).
    Only creates rules for variables that have both type_of and range_of populated.
    """
    rules = {}  # key: actual column name, value: rule dict
    skipped_invalid_range = 0
    matched_count = 0
    
    for _, row in lookup_with_rules.iterrows():
        base_var = row['variable_base']
        type_of = row['type_of']
        range_of = row['range_of']
        
        # Skip if base variable name is missing (shouldn't happen with our filter)
        if pd.isna(base_var):
            continue
        
        # Parse the range
        parsed_range = parse_range(range_of)
        if parsed_range is None or len(parsed_range) == 0:
            skipped_invalid_range += 1
            continue
        
        # Determine rule type and values
        type_str = str(type_of).lower()
        if 'continuous' in type_str:
            # For continuous, we need min/max as numbers (could be int or float)
            numeric_values = [v for v in parsed_range if isinstance(v, (int, float))]
            if len(numeric_values) >= 2:
                rule = {
                    'type': 'continuous',
                    'min': min(numeric_values),
                    'max': max(numeric_values),
                    'expected_range': parsed_range,
                    'base_var': base_var
                }
            else:
                skipped_invalid_range += 1
                continue
        elif 'discrete' in type_str:
            # For discrete, keep the values as they are (preserving ints)
            rule = {
                'type': 'discrete',
                'values': parsed_range,
                'expected_values': parsed_range,
                'base_var': base_var
            }
        else:
            # Skip unknown types
            continue
        
        # Find all columns in dataset that match this base name as a prefix
        # Case-insensitive, exact prefix match (starts with base_var)
        matching_cols = [col for col in df.columns 
                        if col.upper().startswith(base_var.upper())]
        
        if matching_cols:
            matched_count += 1
            for col in matching_cols:
                if col not in rules:
                    rules[col] = rule
                else:
                    # Keep first rule, but warn
                    print(f"Warning: {col} already has a rule from {rules[col]['base_var']}, skipping {base_var}")
    
    print(f"\nRule building summary:")
    print(f"  - Lookup rows with validation rules: {len(lookup_with_rules)}")
    print(f"  - Lookup bases that matched dataset columns: {matched_count}")
    print(f"  - Dataset columns with rules created: {len(rules)}")
    print(f"  - Skipped (invalid range format): {skipped_invalid_range}")
    
    return rules


In [112]:
# Build the rules (only from lookup entries that have both type and range)
print("Building rules with prefix matching...")
print(f"Using {len(lookup_with_rules)} lookup entries that have validation rules")
rules = build_rules_with_prefix(df, lookup_with_rules)

print(f"\n✓ Created rules for {len(rules)} dataset columns")

# Show some examples
if len(rules) > 0:
    print("\nExample rules (first 15):")
    for i, (col, rule) in enumerate(list(rules.items())[:15]):
        rule_type = rule['type']
        if rule_type == 'continuous':
            print(f"  {col}: {rule_type} [{rule['min']} - {rule['max']}] (type: {type(rule['min']).__name__})")
        else:
            values = rule['values'][:5]
            # Show the type of the first value to demonstrate integer preservation
            val_types = set(type(v).__name__ for v in rule['values'][:3])
            values_str = str(values) + ('...' if len(rule['values']) > 5 else '')
            print(f"  {col}: {rule_type} {values_str} (types: {val_types})")


Building rules with prefix matching...
Using 89 lookup entries that have validation rules

Rule building summary:
  - Lookup rows with validation rules: 89
  - Lookup bases that matched dataset columns: 72
  - Dataset columns with rules created: 75
  - Skipped (invalid range format): 0

✓ Created rules for 75 dataset columns

Example rules (first 15):
  PD_ONSET_SYMPTOM_YEAR_E1_C3: continuous [1935 - 2025] (type: int)
  PD_ONSET_DIAGNOSIS_YEAR_E1_C3: continuous [2015 - 2021] (type: int)
  PD_ONSET_ANTPRKS_YEAR_E1_C3: continuous [1935 - 2025] (type: int)
  PD_ONSET_LDOPA_YEAR_E1_C3: continuous [1935 - 2025] (type: int)
  PD_ONSET_CARE_PARTNER_E1_C3: discrete [1, 0] (types: {'int'})
  PD_ONSET_AFFECTED_SIDE_E1_C3: discrete [1, 2, 3, 99] (types: {'int'})
  PD_PROGR_SUBTYPE_E1_C4: discrete [1, 2, 3] (types: {'int'})
  PD_PROGR_HY_E1_C4: discrete [1, 2, 2.5, 3, 4]... (types: {'int', 'float'})
  PD_PROGR_FALLS_E1_C4: discrete [1, 0] (types: {'int'})
  PD_PROGR_FALLS_NUM_E1_C4: discrete [1, 0

In [113]:
def run_qc(df, rules, id_col='Study.Subject.ID'):
    """
    Run quality checks on the dataset
    Returns summary dataframe and audit dataframe
    """
    summary_rows = []
    audit_rows = []
    
    # Get subject IDs
    if id_col not in df.columns:
        print(f"Warning: ID column '{id_col}' not found. Using index.")
        ids = df.index.astype(str)
    else:
        ids = df[id_col]
    
    print(f"\nRunning QC on {len(rules)} variables...")
    
    for idx, (var, rule) in enumerate(rules.items()):
        if var not in df.columns:
            print(f"  Warning: {var} not found in data, skipping")
            continue
        
        if idx % 50 == 0 and idx > 0:
            print(f"  Progress: {idx}/{len(rules)} variables checked...")
            
        x = df[var]
        
        # Check for missing
        missing = is_missing(x)
        missing_pct = missing.mean() * 100
        
        # Convert to numeric - but preserve integers vs floats intelligently
        # First try int, then float
        x_num_int = pd.to_numeric(x, errors='coerce', downcast='integer')
        x_num_float = pd.to_numeric(x, errors='coerce')
        
        # Special handling for year variables
        if is_year_variable(var):
            x_num_int = extract_year(x)
            x_num_float = extract_year(x)
        
        # Determine valid values
        valid = pd.Series([False] * len(df), index=df.index)
        
        if rule['type'] == 'continuous':
            # For continuous, check if within range using float comparison
            valid = (~x_num_float.isna()) & (x_num_float >= rule['min']) & (x_num_float <= rule['max'])
            
        elif rule['type'] == 'discrete':
            # For discrete, we need to compare with the parsed values (which may be ints or floats)
            # Convert both sides to the same type for proper comparison
            valid_mask = pd.Series([False] * len(df), index=df.index)
            
            for expected_val in rule['values']:
                if isinstance(expected_val, int):
                    # Compare with integer conversion
                    mask = (x_num_int == expected_val)
                    valid_mask = valid_mask | mask
                elif isinstance(expected_val, float):
                    # Compare with float conversion
                    mask = (x_num_float == expected_val)
                    valid_mask = valid_mask | mask
                else:
                    # String comparison for non-numeric discrete values
                    mask = (x.astype(str).str.strip() == str(expected_val))
                    valid_mask = valid_mask | mask
            
            valid = (~x_num_float.isna()) & valid_mask
        
        valid_pct = valid.mean() * 100
        out_of_range_pct = ((~valid) & (~missing)).mean() * 100
        
        # Store summary
        summary_rows.append({
            'variable': var,
            'base_variable': rule.get('base_var', ''),
            'rule_type': rule['type'],
            'missing_pct': round(missing_pct, 2),
            'valid_pct': round(valid_pct, 2),
            'out_of_range_pct': round(out_of_range_pct, 2),
            'total_n': len(df),
            'missing_n': missing.sum(),
            'valid_n': valid.sum(),
            'out_of_range_n': ((~valid) & (~missing)).sum()
        })
        
        # Create audit records for failures (limit to first 1000 per variable to avoid memory issues)
        MAX_AUDIT_PER_VAR = 1000
        
        # Missing values
        missing_indices = df.index[missing]
        if len(missing_indices) > 0:
            for idx_val in missing_indices[:MAX_AUDIT_PER_VAR]:
                audit_rows.append({
                    'subject_id': ids.iloc[idx_val],
                    'variable': var,
                    'issue_type': 'MISSING',
                    'value': None,
                    'expected': str(rule.get('expected_range', rule.get('expected_values', ''))),
                    'status': 'FAIL'
                })
        
        # Out of range values
        out_of_range_indices = df.index[(~valid) & (~missing)]
        if len(out_of_range_indices) > 0:
            for idx_val in out_of_range_indices[:MAX_AUDIT_PER_VAR]:
                audit_rows.append({
                    'subject_id': ids.iloc[idx_val],
                    'variable': var,
                    'issue_type': 'OUT_OF_RANGE',
                    'value': x.iloc[idx_val],
                    'expected': str(rule.get('expected_range', rule.get('expected_values', ''))),
                    'status': 'FAIL'
                })
    
    print(f"  ✓ Complete! Processed {len(rules)} variables")
    
    summary_df = pd.DataFrame(summary_rows)
    audit_df = pd.DataFrame(audit_rows)
    
    return summary_df, audit_df


In [114]:
print("=" * 80)
print("RUNNING QC CHECKS")
print("=" * 80)

if len(rules) == 0:
    print("No validation rules created! Please check the lookup file format.")
else:
    qc_summary, qc_audit = run_qc(df, rules)
    
    print(f"\n✓ QC Complete!")
    print(f"  - Summary records: {len(qc_summary)}")
    print(f"  - Audit records: {len(qc_audit)}")


RUNNING QC CHECKS

Running QC on 75 variables...
  Progress: 50/75 variables checked...
  ✓ Complete! Processed 75 variables

✓ QC Complete!
  - Summary records: 75
  - Audit records: 205


In [115]:
if len(rules) > 0:
    # Summary of QC results
    print("\n" + "=" * 80)
    print("QC SUMMARY - First 20 variables")
    print("=" * 80)
    qc_summary.head(20)


QC SUMMARY - First 20 variables


In [116]:
if len(rules) > 0:
    # Variables with highest missing rates
    print("\n" + "=" * 80)
    print("TOP 10 VARIABLES WITH HIGHEST MISSING RATES")
    print("=" * 80)
    qc_summary_sorted = qc_summary.sort_values('missing_pct', ascending=False)
    qc_summary_sorted[['variable', 'missing_pct', 'valid_pct', 'out_of_range_pct']].head(10)


TOP 10 VARIABLES WITH HIGHEST MISSING RATES


In [117]:
if len(rules) > 0:
    # Variables with highest out-of-range rates (excluding high missing)
    print("\n" + "=" * 80)
    print("TOP 10 VARIABLES WITH HIGHEST OUT-OF-RANGE RATES")
    print("=" * 80)
    qc_with_data = qc_summary[qc_summary['valid_pct'] < 100]
    if len(qc_with_data) > 0:
        qc_sorted_or = qc_with_data.sort_values('out_of_range_pct', ascending=False)
        qc_sorted_or[['variable', 'out_of_range_pct', 'missing_pct', 'valid_pct']].head(10)
    else:
        print("No out-of-range values found!")


TOP 10 VARIABLES WITH HIGHEST OUT-OF-RANGE RATES


In [118]:
if len(rules) > 0 and len(qc_audit) > 0:
    # Audit trail sample
    print("\n" + "=" * 80)
    print("AUDIT TRAIL SAMPLE (first 20 failures)")
    print("=" * 80)
    qc_audit.head(20)
    
    # Summary statistics of audits
    print("\n" + "=" * 80)
    print("AUDIT SUMMARY BY ISSUE TYPE")
    print("=" * 80)
    audit_summary = qc_audit['issue_type'].value_counts()
    print(audit_summary)
    
    print("\n" + "=" * 80)
    print("TOP 10 VARIABLES WITH MOST AUDIT FAILURES")
    print("=" * 80)
    qc_audit['variable'].value_counts().head(10)
elif len(rules) > 0:
    print("\n✓ No audit failures found! All validated variables passed QC.")



AUDIT TRAIL SAMPLE (first 20 failures)

AUDIT SUMMARY BY ISSUE TYPE
issue_type
MISSING         180
OUT_OF_RANGE     25
Name: count, dtype: int64

TOP 10 VARIABLES WITH MOST AUDIT FAILURES


In [119]:
qc_summary

,variable,base_variable,rule_type,missing_pct,valid_pct,out_of_range_pct,total_n,missing_n,valid_n,out_of_range_n
0,PD_ONSET_SYMPTOM_YEAR_E1_C3,PD_ONSET_SYMPTOM_YEAR,continuous,0.00,100.00,0.00,30,0,30,0
1,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,PD_ONSET_DIAGNOSIS_YEAR,continuous,0.00,76.67,23.33,30,0,23,7
2,PD_ONSET_ANTPRKS_YEAR_E1_C3,PD_ONSET_ANTPRKS_YEAR,continuous,0.00,100.00,0.00,30,0,30,0
3,PD_ONSET_LDOPA_YEAR_E1_C3,PD_ONSET_LDOPA_YEAR,continuous,0.00,100.00,0.00,30,0,30,0
4,PD_ONSET_CARE_PARTNER_E1_C3,PD_ONSET_CARE_PARTNER,discrete,0.00,100.00,0.00,30,0,30,0
...,...,...,...,...,...,...,...,...,...,...
70,PD_MED_LEV_UNIT_DAY_E1_C8_1,PD_MED_LEV_UNIT_DAY,continuous,0.00,83.33,16.67,30,0,25,5
71,PD_MED_LEV_UNIT_DAY_E1_C8_2,PD_MED_LEV_UNIT_DAY,continuous,76.67,23.33,0.00,30,23,7,0
72,PD_MED_LEV_UNIT_DAY_E1_C8_3,PD_MED_LEV_UNIT_DAY,continuous,93.33,6.67,0.00,30,28,2,0
73,PD_MED_LEV_UNIT_DAY_E1_C8_4,PD_MED_LEV_UNIT_DAY,continuous,96.67,3.33,0.00,30,29,1,0


In [ ]:
qc_audit

In [120]:
# Export to CSV files
output_prefix = "AI_PMP_qc_output_2026_04_25"
qc_summary.to_csv(f"{output_prefix}_summary.csv", index=False)
qc_audit.to_csv(f"{output_prefix}_audit.csv", index=False)

In [121]:
qc_audit

,subject_id,variable,issue_type,value,expected,status
0,fr02009,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,OUT_OF_RANGE,2022,"[2015, 2021]",FAIL
1,fr02011,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,OUT_OF_RANGE,2022,"[2015, 2021]",FAIL
2,fr02016,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,OUT_OF_RANGE,2022,"[2015, 2021]",FAIL
3,fr02017,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,OUT_OF_RANGE,2022,"[2015, 2021]",FAIL
4,es02003,PD_ONSET_DIAGNOSIS_YEAR_E1_C3,OUT_OF_RANGE,2023,"[2015, 2021]",FAIL
...,...,...,...,...,...,...
200,es02005,PD_MED_LEV_UNIT_DAY_E1_C8_4,MISSING,None,"[0, 5]",FAIL
201,es02006,PD_MED_LEV_UNIT_DAY_E1_C8_4,MISSING,None,"[0, 5]",FAIL
202,es02007,PD_MED_LEV_UNIT_DAY_E1_C8_4,MISSING,None,"[0, 5]",FAIL
203,es02009,PD_MED_LEV_UNIT_DAY_E1_C8_4,MISSING,None,"[0, 5]",FAIL


In [123]:
# Check which lookup bases didn't match any columns
lookup_bases =  lookup_with_rules['variable_base'].dropna().unique()
matched_bases = set(rules[col]['base_var'] for col in rules.keys())
unmatched_bases = set(lookup_bases) - matched_bases

print("=" * 80)
print(f"LOOKUP BASES THAT DIDN'T MATCH ANY COLUMNS ({len(unmatched_bases)})")
print("=" * 80)
for base in sorted(unmatched_bases)[:20]:
    print(f"  - {base}")
if len(unmatched_bases) > 20:
    print(f"  ... and {len(unmatched_bases) - 20} more")

LOOKUP BASES THAT DIDN'T MATCH ANY COLUMNS (12)
  - BS_PRS
  - DEM_HEIGHT
  - DEM_WEIGHT
  - MDS_UPDRS_HY_C1
  - MDS_UPDRS_HY_C2
  - MI_INTER_PD_MED_CHANGE
  - MI_INTER_WORS_Q
  - MI_INTER_WORS_Y_MS
  - MI_INTER_WORS_Y_MS_YN
  - MI_INTER_WORS_Y_NMS
  - MI_INTER_WORS_Y_NMS_YN
  - PD_PROGR_FALLS_NUM


In [124]:
set(lookup_bases)

{'BAI_TOTAL',
 'BDI_II_TOTAL',
 'BP_LYING_DBP',
 'BP_LYING_SBP',
 'BP_STANDING_DBP',
 'BP_STANDING_SBP',
 'BS_PRS',
 'BS_TAKEN',
 'CCI_AGE',
 'CCI_TOTAL',
 'CLIN_PRED_Q1',
 'CLIN_PRED_Q2',
 'CLIN_PRED_Q3',
 'CLIN_PRED_Q5_1',
 'CLIN_PRED_Q5_2',
 'CLIN_PRED_Q5_3',
 'CLIN_PRED_Q5_4',
 'CLIN_PRED_Q5_5',
 'DEM_DOB',
 'DEM_EDUCATION',
 'DEM_ETHNICITY',
 'DEM_GENDER',
 'DEM_HANDEDNESS',
 'DEM_HEIGHT',
 'DEM_RACIAL_G',
 'DEM_RESIDENCE',
 'DEM_WEIGHT',
 'ESS_TOTAL',
 'ICF_0',
 'ICF_2',
 'ICF_3',
 'ICF_DATE',
 'INEX_DATE',
 'INEX_EXC_1',
 'INEX_EXC_2',
 'INEX_EXC_3',
 'INEX_EXC_4',
 'INEX_EXC_5',
 'INEX_EXC_6',
 'INEX_EXC_7',
 'INEX_EXC_8',
 'INEX_EXC_9',
 'INEX_INC_1',
 'INEX_INC_2',
 'INEX_INC_3',
 'INEX_INC_4',
 'INEX_INC_5',
 'INEX_INC_6',
 'MDS_UPDRS_1_TOTAL',
 'MDS_UPDRS_2_TOTAL',
 'MDS_UPDRS_3_TOTAL',
 'MDS_UPDRS_4_TOTAL',
 'MDS_UPDRS_ALL_TOTAL',
 'MDS_UPDRS_HY_C1',
 'MDS_UPDRS_HY_C2',
 'MI_INTER_PD_MED_CHANGE',
 'MI_INTER_WORS_Q',
 'MI_INTER_WORS_Y_MS',
 'MI_INTER_WORS_Y_MS_YN',
 'MI_INT